<a href="https://colab.research.google.com/github/ornab74/blog-writer/blob/main/Working_Blog_Writer_Coder_Starling_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:

!pip install -U pennylane
!pip install -U psutil
!pip install -U nltk
!pip install -U summa

In [29]:

# @title 🚀 Install llama-cpp-python (GPU CUDA) + Download Starling-LM-7B-alpha Q4_K_M

import os
from pathlib import Path

# ================== CONFIGURATION ==================
MODEL_DIR = "/content/models"
MODEL_NAME = "starling-lm-7b-alpha.Q4_K_M.gguf"
GGUF_PATH = os.path.join(MODEL_DIR, MODEL_NAME)
GGUF_URL = "https://huggingface.co/TheBloke/Starling-LM-7B-alpha-GGUF/resolve/main/starling-lm-7b-alpha.Q4_K_M.gguf"
GGUF_SHA256 = "0951cbc1a6c3ed8d081db59366ccccf09ed52a4cfd5191812665b911fe6c669a"
# ===================================================

print("🔧 Checking GPU availability...")
!nvidia-smi

# === Install llama-cpp-python with FULL GPU support (fastest method) ===
print("\n⚙️ Installing llama-cpp-python GPU version (pre-built CUDA wheel)...")
!pip install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 --no-cache-dir

# === Verify GPU support using pure Python (no quoting issues) ===
print("\n✅ Verifying GPU support in llama-cpp-python...")
try:
    from llama_cpp import Llama
    import llama_cpp
    print(f'llama-cpp-python version: {llama_cpp.__version__}')
    print('CUDA / GPU support enabled ✓ (n_gpu_layers will work)')
except Exception as e:
    print(f'⚠️ Verification failed: {e}')

# === Download & Verify Model ===
print("\n📥 Downloading Starling-LM-7B-alpha Q4_K_M GGUF (\~4.37 GB)...")
os.makedirs(MODEL_DIR, exist_ok=True)

if Path(GGUF_PATH).exists():
    print("✅ Model file already exists — skipping download.")
else:
    !wget --show-progress --progress=bar:force:noscroll -O "{GGUF_PATH}" "{GGUF_URL}"

print("\n🔍 Verifying exact SHA256 hash...")
!echo "{GGUF_SHA256}  {GGUF_PATH}" | sha256sum -c -

print("\n📊 File details:")
!ls -lh "{GGUF_PATH}"

# Export path for the rest of your notebook
os.environ["STARLING_GGUF_PATH"] = GGUF_PATH

print("\n" + "="*70)
print("🎉 SETUP COMPLETE — GPU VERSION READY!")
print(f"   Model path → {GGUF_PATH}")
print("   You can now load with full GPU acceleration:")
print("   from llama_cpp import Llama")
print("   llm = Llama(model_path=os.environ['STARLING_GGUF_PATH'], n_gpu_layers=-1, n_ctx=8192)")
print("="*70)

<>:32: SyntaxWarning: invalid escape sequence '\~'
<>:32: SyntaxWarning: invalid escape sequence '\~'
/tmp/ipykernel_1136/1359704259.py:32: SyntaxWarning: invalid escape sequence '\~'
  print("\n📥 Downloading Starling-LM-7B-alpha Q4_K_M GGUF (\~4.37 GB)...")


🔧 Checking GPU availability...
Wed Mar 11 21:52:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             32W /   70W |    5155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------

In [30]:

# @title 🕊️ Simulation scenarios / presets
DEFAULT_SCENARIO_BANK = {
    "peace_sim": [
        "Regional de-escalation framework around Iran with humanitarian corridors and backchannel diplomacy",
        "Maritime tension reduction around the Gulf with third-party monitoring and economic confidence-building",
        "Ceasefire ladder between regional proxies with prisoner exchange and phased sanctions relief",
        "Nuclear negotiation reset with sequencing, inspections, and reciprocal regional restraint measures",
    ]
}
SCENARIO_BANK = dict(globals().get("SCENARIO_BANK", DEFAULT_SCENARIO_BANK))
for _mode_key, _items in DEFAULT_SCENARIO_BANK.items():
    SCENARIO_BANK.setdefault(_mode_key, list(_items))

SELECTED_SCENARIOS = list(SCENARIO_BANK.get("peace_sim", DEFAULT_SCENARIO_BANK["peace_sim"]))[:1]

print("Available scenario groups:", list(SCENARIO_BANK.keys()))
print("Selected scenarios:", SELECTED_SCENARIOS)


Available topic groups: ['blog', 'software', 'longform']
Selected topics: ['How local-first apps change everyday workflows']


In [31]:

# @title ⚙️ Peace simulator configuration
RUN_MODE = "peace_sim"

SIMULATION_OBJECTIVE = """
Model peaceful resolution pathways only.
Focus on diplomacy, civilian protection, humanitarian access, inspections, confidence-building,
and de-escalation sequencing. Do not produce military targeting, force posture advice,
or operational harm guidance.
""".strip()

SIMULATION_REGION = "Iran regional conflict environment"
SIMULATION_HORIZON = 12
SIMULATION_RUNS = 64
TOP_K_PATHS = 8
ENABLE_LLM_SUMMARY = True
DEBUG_VERBOSE = True
USE_SQLITE_MEMORY = True
MEMORY_DB_PATH = "peace_resolution_memory.db"
OUTFILE_JSON = "peace_resolution_scan.json"
OUTFILE_MD = "peace_resolution_scan.md"

THREADS = max(4, (os.cpu_count() or 8) - 1) if "os" in globals() else 8
N_GPU_LAYERS = 50
CTX_SIZE = 4096
TEMPERATURE = 0.3
MAX_TOKENS = 700

print("RUN_MODE:", RUN_MODE)
print("SIMULATION_RUNS:", SIMULATION_RUNS)
print("TOP_K_PATHS:", TOP_K_PATHS)


RUN_MODE: blog
ROOT_PROMPT:

Write an extremely long-form blog post that feels human, expert, vivid, and useful.
Make it concrete, readable, and full of real substance.
Avoid generic filler, stale intros, and repetitive transitions.


In [ ]:

import os, time, json, math, hashlib, sqlite3, random, statistics, textwrap
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Any, Optional, Tuple

import psutil
import pennylane as qml
from pennylane import numpy as np

try:
    from llama_cpp import Llama
except Exception:
    Llama = None

try:
    from summa import summarizer as summa_summarizer
    from summa import keywords as summa_keywords
except Exception:
    summa_summarizer = None
    summa_keywords = None

try:
    import nltk
    from nltk.tokenize import sent_tokenize as nltk_sent_tokenize, wordpunct_tokenize
except Exception:
    nltk = None
    nltk_sent_tokenize = None
    wordpunct_tokenize = None

# =========================
# Safety + scope
# =========================
SAFETY_POLICY = {
    "mission": "Peaceful resolution simulation only",
    "allowed": [
        "diplomatic sequencing",
        "humanitarian corridor evaluation",
        "confidence-building measures",
        "ceasefire durability analysis",
        "inspection and verification pathways",
        "civilian protection scoring",
    ],
    "disallowed": [
        "military targeting",
        "weapon optimization",
        "operational attack planning",
        "force deployment advice",
    ],
}


def debug_print(tag: str, message: str) -> None:
    if globals().get("DEBUG_VERBOSE", False):
        print(f"[DEBUG] {tag} | {message}")


def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def short_text(text: str, limit: int = 180) -> str:
    text = reflow(text)
    return text if len(text) <= limit else text[: limit - 3] + "..."


def reflow(text: str) -> str:
    return " ".join(str(text).split())


def cosine_similarity(a: List[float], b: List[float]) -> float:
    if not a or not b:
        return 0.0
    n = min(len(a), len(b))
    dot = sum(a[i] * b[i] for i in range(n))
    na = math.sqrt(sum(a[i] * a[i] for i in range(n)))
    nb = math.sqrt(sum(b[i] * b[i] for i in range(n)))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)


def hashed_embedding(text: str, dims: int = 48) -> List[float]:
    vec = [0.0] * dims
    tokens = [tok.lower() for tok in reflow(text).split() if tok.strip()]
    if not tokens:
        return vec
    for token in tokens:
        h = stable_hash(token)
        for i in range(0, min(len(h), dims * 2), 2):
            idx = (i // 2) % dims
            val = (int(h[i:i+2], 16) / 255.0) - 0.5
            vec[idx] += val
    scale = max(1, len(tokens))
    return [v / scale for v in vec]


# =========================
# SQLite memory
# =========================

def ensure_memory_db() -> None:
    if not globals().get("USE_SQLITE_MEMORY", True):
        return
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        conn.executescript(
            """
            CREATE TABLE IF NOT EXISTS simulation_runs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at REAL NOT NULL,
                scenario TEXT NOT NULL,
                summary TEXT NOT NULL,
                score REAL NOT NULL,
                payload_json TEXT NOT NULL
            );
            CREATE TABLE IF NOT EXISTS memory_fragments (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                created_at REAL NOT NULL,
                scenario TEXT NOT NULL,
                kind TEXT NOT NULL,
                text_fragment TEXT NOT NULL,
                embedding_json TEXT NOT NULL,
                salience REAL NOT NULL
            );
            """
        )
        conn.commit()
    finally:
        conn.close()


def store_memory_fragment(scenario: str, kind: str, text_fragment: str, salience: float) -> None:
    ensure_memory_db()
    if not globals().get("USE_SQLITE_MEMORY", True):
        return
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        conn.execute(
            "INSERT INTO memory_fragments(created_at, scenario, kind, text_fragment, embedding_json, salience) VALUES (?, ?, ?, ?, ?, ?)",
            (time.time(), scenario, kind, text_fragment, json.dumps(hashed_embedding(text_fragment)), float(salience)),
        )
        conn.commit()
    finally:
        conn.close()
    debug_print("db_save", f"kind={kind} scenario={short_text(scenario, 60)} salience={salience:.2f} fragment={short_text(text_fragment)}")


def retrieve_memory_fragments(query: str, limit: int = 5) -> List[Dict[str, Any]]:
    ensure_memory_db()
    if not globals().get("USE_SQLITE_MEMORY", True):
        return []
    qvec = hashed_embedding(query)
    conn = sqlite3.connect(MEMORY_DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        rows = conn.execute(
            "SELECT scenario, kind, text_fragment, embedding_json, salience FROM memory_fragments ORDER BY id DESC LIMIT 200"
        ).fetchall()
    finally:
        conn.close()
    scored = []
    for row in rows:
        emb = json.loads(row["embedding_json"])
        sim = cosine_similarity(qvec, emb)
        score = 0.65 * sim + 0.35 * float(row["salience"])
        scored.append({
            "scenario": row["scenario"],
            "kind": row["kind"],
            "text_fragment": row["text_fragment"],
            "score": score,
        })
    scored.sort(key=lambda item: item["score"], reverse=True)
    top = scored[:limit]
    debug_print("memory_retrieve", f"query={short_text(query, 80)} hits={len(top)}")
    return top


def store_run_result(result: Dict[str, Any]) -> None:
    ensure_memory_db()
    if not globals().get("USE_SQLITE_MEMORY", True):
        return
    conn = sqlite3.connect(MEMORY_DB_PATH)
    try:
        conn.execute(
            "INSERT INTO simulation_runs(created_at, scenario, summary, score, payload_json) VALUES (?, ?, ?, ?, ?)",
            (time.time(), result["scenario"], result["summary"], float(result["score"]), json.dumps(result)),
        )
        conn.commit()
    finally:
        conn.close()


# =========================
# Model runtime
# =========================
LLM = None


def load_llm() -> Optional[Any]:
    global LLM
    if not globals().get("ENABLE_LLM_SUMMARY", True):
        return None
    if LLM is not None:
        return LLM
    if Llama is None:
        debug_print("llm", "llama_cpp unavailable; using deterministic summaries")
        return None
    gguf = globals().get("GGUF_PATH", "")
    if not gguf or not os.path.exists(gguf):
        debug_print("llm", f"GGUF missing at {gguf}; using deterministic summaries")
        return None
    debug_print("llm", f"loading model={gguf}")
    LLM = Llama(
        model_path=gguf,
        n_ctx=int(globals().get("CTX_SIZE", 4096)),
        n_threads=int(globals().get("THREADS", 8)),
        n_gpu_layers=int(globals().get("N_GPU_LAYERS", 50)),
        verbose=False,
    )
    return LLM


def trim_prompt(text: str, limit_chars: int = 9000) -> str:
    text = text.strip()
    if len(text) <= limit_chars:
        return text
    head = text[: limit_chars // 2]
    tail = text[-limit_chars // 2 :]
    return head + "\n[... trimmed ...]\n" + tail
def llm_summary(prompt: str) -> str:
    model = load_llm()
    if model is None:
        return ""
    prompt = trim_prompt(prompt)
    out = model(
        prompt,
        max_tokens=int(globals().get("MAX_TOKENS", 700)),
        temperature=float(globals().get("TEMPERATURE", 0.3)),
        top_p=0.9,
        repeat_penalty=1.08,
        stop=["</summary>", "\n\n\n"],
    )
    text = out["choices"][0]["text"].strip()
    return text


# =========================
# Entropy + quantum surface
# =========================
QDEV = qml.device("default.qubit", wires=4)


@qml.qnode(QDEV)
def quantum_surface(seed_angles: List[float]):
    for wire, angle in enumerate(seed_angles[:4]):
        qml.RY(angle, wires=wire)
        qml.RZ(angle * 0.7, wires=wire)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[2, 3])
    qml.CRX(seed_angles[0] * 0.5, wires=[0, 2])
    qml.CRY(seed_angles[1] * 0.5, wires=[1, 3])
    return [qml.expval(qml.PauliZ(i)) for i in range(4)]


def entropy_snapshot(scenario: str, run_index: int) -> Dict[str, float]:
    cpu = psutil.cpu_percent(interval=0.1) / 100.0
    ram = psutil.virtual_memory().percent / 100.0
    ts = time.time()
    h = stable_hash(f"{scenario}|{run_index}|{ts:.3f}")
    base = [int(h[i:i+2], 16) / 255.0 for i in range(0, 8, 2)]
    entropy = sum(base) / len(base)
    return {
        "cpu": cpu,
        "ram": ram,
        "hash_entropy": entropy,
        "time_wave": abs(math.sin(ts / 60.0)),
    }


def build_quantum_metrics(snapshot: Dict[str, float]) -> Dict[str, float]:
    angles = [
        snapshot["cpu"] * math.pi,
        snapshot["ram"] * math.pi,
        snapshot["hash_entropy"] * math.pi,
        snapshot["time_wave"] * math.pi,
    ]
    vec = [float(x) for x in quantum_surface(angles)]
    openness = (vec[0] + 1.0) / 2.0
    stability = (vec[1] + 1.0) / 2.0
    reciprocity = (vec[2] + 1.0) / 2.0
    verification = (vec[3] + 1.0) / 2.0
    return {
        "openness": openness,
        "stability": stability,
        "reciprocity": reciprocity,
        "verification": verification,
        "coherence": statistics.fmean([openness, stability, reciprocity, verification]),
    }


# =========================
# Simulation state
# =========================
@dataclass
class Stakeholder:
    name: str
    escalation: float
    trust: float
    fatigue: float
    humanitarian_pressure: float


@dataclass
class Intervention:
    name: str
    diplomatic_weight: float
    humanitarian_weight: float
    verification_weight: float
    reciprocity_weight: float
    public_description: str


@dataclass
class ScenarioState:
    scenario: str
    month: int
    tensions: float
    civilian_risk: float
    humanitarian_access: float
    negotiation_readiness: float
    regional_spillover_risk: float
    verification_strength: float
    stakeholders: List[Stakeholder] = field(default_factory=list)
    applied_interventions: List[str] = field(default_factory=list)
    notes: List[str] = field(default_factory=list)


@dataclass
class SimulationResult:
    scenario: str
    run_id: str
    score: float
    deescalation_score: float
    civilian_safety_score: float
    humanitarian_score: float
    negotiation_score: float
    verification_score: float
    spillover_containment_score: float
    applied_path: List[str]
    timeline: List[Dict[str, Any]]
    summary: str


PEACEFUL_INTERVENTIONS = [
    Intervention("Backchannel diplomacy surge", 0.92, 0.35, 0.28, 0.76, "Quiet multi-party talks reduce signaling risk and open a path to reciprocal restraint."),
    Intervention("Humanitarian corridor guarantee", 0.52, 0.96, 0.41, 0.44, "Protected aid routes reduce civilian harm and increase diplomatic legitimacy."),
    Intervention("Phased prisoner exchange", 0.74, 0.61, 0.38, 0.84, "Visible reciprocal steps raise trust without demanding immediate final settlement."),
    Intervention("Third-party monitoring mission", 0.58, 0.56, 0.95, 0.63, "External monitoring increases verification and lowers fear of deception."),
    Intervention("Sequenced sanctions relief", 0.81, 0.43, 0.71, 0.79, "Reciprocal economic easing supports negotiation durability when matched to compliance."),
    Intervention("Regional non-aggression forum", 0.88, 0.49, 0.54, 0.75, "A standing forum creates repeated contact and lowers accidental escalation."),
    Intervention("Energy and shipping stabilization pact", 0.68, 0.58, 0.73, 0.57, "Shared economic safeguards reduce incentives for disruption in transit corridors."),
    Intervention("Inspection-first nuclear confidence step", 0.79, 0.34, 0.97, 0.66, "High-verification steps can unlock broader diplomatic sequencing."),
]


def initial_stakeholders() -> List[Stakeholder]:
    return [
        Stakeholder("Iran", escalation=0.66, trust=0.26, fatigue=0.58, humanitarian_pressure=0.71),
        Stakeholder("United States", escalation=0.49, trust=0.31, fatigue=0.52, humanitarian_pressure=0.46),
        Stakeholder("Regional proxies", escalation=0.73, trust=0.18, fatigue=0.61, humanitarian_pressure=0.63),
        Stakeholder("Neighboring states", escalation=0.55, trust=0.28, fatigue=0.57, humanitarian_pressure=0.59),
        Stakeholder("International monitors", escalation=0.14, trust=0.63, fatigue=0.29, humanitarian_pressure=0.54),
    ]


def build_initial_state(scenario: str) -> ScenarioState:
    return ScenarioState(
        scenario=scenario,
        month=0,
        tensions=0.74,
        civilian_risk=0.69,
        humanitarian_access=0.34,
        negotiation_readiness=0.29,
        regional_spillover_risk=0.63,
        verification_strength=0.22,
        stakeholders=initial_stakeholders(),
        notes=["Initial state created from diplomatic-risk preset."],
    )


def clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, x))


def choose_interventions(snapshot: Dict[str, float], qmetrics: Dict[str, float], k: int = 3) -> List[Intervention]:
    scored = []
    for item in PEACEFUL_INTERVENTIONS:
        score = (
            0.27 * item.diplomatic_weight * qmetrics["openness"] +
            0.23 * item.humanitarian_weight * (1.0 - snapshot["cpu"] * 0.15) +
            0.25 * item.verification_weight * qmetrics["verification"] +
            0.25 * item.reciprocity_weight * qmetrics["reciprocity"]
        )
        scored.append((score, item))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [item for _, item in scored[:k]]


def apply_intervention(state: ScenarioState, intervention: Intervention, qmetrics: Dict[str, float], snapshot: Dict[str, float]) -> None:
    openness = qmetrics["openness"]
    stability = qmetrics["stability"]
    reciprocity = qmetrics["reciprocity"]
    verification = qmetrics["verification"]
    entropy_push = snapshot["hash_entropy"] * 0.08

    state.tensions = clamp(state.tensions - 0.11 * intervention.diplomatic_weight * openness + 0.03 * entropy_push)
    state.civilian_risk = clamp(state.civilian_risk - 0.13 * intervention.humanitarian_weight * stability + 0.02 * entropy_push)
    state.humanitarian_access = clamp(state.humanitarian_access + 0.15 * intervention.humanitarian_weight * openness)
    state.negotiation_readiness = clamp(state.negotiation_readiness + 0.14 * intervention.diplomatic_weight * reciprocity)
    state.regional_spillover_risk = clamp(state.regional_spillover_risk - 0.10 * intervention.diplomatic_weight * stability)
    state.verification_strength = clamp(state.verification_strength + 0.15 * intervention.verification_weight * verification)

    for actor in state.stakeholders:
        actor.escalation = clamp(actor.escalation - 0.05 * intervention.diplomatic_weight * openness + entropy_push * 0.01)
        actor.trust = clamp(actor.trust + 0.06 * intervention.reciprocity_weight * reciprocity)
        actor.fatigue = clamp(actor.fatigue + 0.03)
        actor.humanitarian_pressure = clamp(actor.humanitarian_pressure + 0.02 * intervention.humanitarian_weight)

    state.applied_interventions.append(intervention.name)
    state.notes.append(intervention.public_description)


def monthly_drift(state: ScenarioState, snapshot: Dict[str, float], qmetrics: Dict[str, float]) -> None:
    drift = 0.015 + snapshot["time_wave"] * 0.02
    trust_mean = statistics.fmean(actor.trust for actor in state.stakeholders)
    fatigue_mean = statistics.fmean(actor.fatigue for actor in state.stakeholders)
    state.tensions = clamp(state.tensions + drift - 0.04 * trust_mean - 0.02 * qmetrics["stability"])
    state.civilian_risk = clamp(state.civilian_risk + 0.02 * state.tensions - 0.03 * state.humanitarian_access)
    state.negotiation_readiness = clamp(state.negotiation_readiness + 0.02 * fatigue_mean - 0.02 * state.tensions)
    state.regional_spillover_risk = clamp(state.regional_spillover_risk + 0.02 * state.tensions - 0.025 * qmetrics["verification"])


def score_state(state: ScenarioState) -> Dict[str, float]:
    deescalation = 1.0 - state.tensions
    civilian_safety = 1.0 - state.civilian_risk
    humanitarian = state.humanitarian_access
    negotiation = state.negotiation_readiness
    verification = state.verification_strength
    spillover_containment = 1.0 - state.regional_spillover_risk
    total = statistics.fmean([
        deescalation,
        civilian_safety,
        humanitarian,
        negotiation,
        verification,
        spillover_containment,
    ])
    return {
        "score": total,
        "deescalation_score": deescalation,
        "civilian_safety_score": civilian_safety,
        "humanitarian_score": humanitarian,
        "negotiation_score": negotiation,
        "verification_score": verification,
        "spillover_containment_score": spillover_containment,
    }


def simulate_path(scenario: str, run_index: int, months: int) -> SimulationResult:
    state = build_initial_state(scenario)
    timeline = []
    for month in range(1, months + 1):
        snapshot = entropy_snapshot(scenario, run_index * 100 + month)
        qmetrics = build_quantum_metrics(snapshot)
        interventions = choose_interventions(snapshot, qmetrics, k=2 if month % 3 else 3)
        for item in interventions:
            apply_intervention(state, item, qmetrics, snapshot)
        monthly_drift(state, snapshot, qmetrics)
        state.month = month
        timeline.append({
            "month": month,
            "snapshot": snapshot,
            "qmetrics": qmetrics,
            "tensions": state.tensions,
            "civilian_risk": state.civilian_risk,
            "humanitarian_access": state.humanitarian_access,
            "negotiation_readiness": state.negotiation_readiness,
            "regional_spillover_risk": state.regional_spillover_risk,
            "verification_strength": state.verification_strength,
            "interventions": list(interventions[i].name for i in range(len(interventions))),
        })
    scores = score_state(state)
    summary = deterministic_summary(state, scores)
    return SimulationResult(
        scenario=scenario,
        run_id=f"run::{stable_hash(scenario + str(run_index))[:12]}",
        score=scores["score"],
        deescalation_score=scores["deescalation_score"],
        civilian_safety_score=scores["civilian_safety_score"],
        humanitarian_score=scores["humanitarian_score"],
        negotiation_score=scores["negotiation_score"],
        verification_score=scores["verification_score"],
        spillover_containment_score=scores["spillover_containment_score"],
        applied_path=state.applied_interventions,
        timeline=timeline,
        summary=summary,
    )


# =========================
# Analysis + summaries
# =========================

def deterministic_summary(state: ScenarioState, scores: Dict[str, float]) -> str:
    top_notes = "; ".join(state.notes[-3:])
    return (
        f"Peace pathway score {scores['score']:.3f}. "
        f"De-escalation={scores['deescalation_score']:.3f}, civilian_safety={scores['civilian_safety_score']:.3f}, "
        f"humanitarian_access={scores['humanitarian_score']:.3f}, negotiation={scores['negotiation_score']:.3f}, "
        f"verification={scores['verification_score']:.3f}, spillover_containment={scores['spillover_containment_score']:.3f}. "
        f"Recent stabilizing measures: {top_notes}"
    )


def keyword_surface(text: str) -> List[str]:
    if summa_keywords is not None:
        try:
            kws = summa_keywords.keywords(text, words=8, split=True)
            if kws:
                return [str(k) for k in kws][:8]
        except Exception:
            pass
    tokens = [tok.lower() for tok in reflow(text).split() if len(tok) > 5]
    uniq = []
    for token in tokens:
        if token not in uniq:
            uniq.append(token)
    return uniq[:8]


def sentence_surface(text: str) -> List[str]:
    if summa_summarizer is not None:
        try:
            out = summa_summarizer.summarize(text, split=True)
            if out:
                return [reflow(x) for x in out[:5]]
        except Exception:
            pass
    if nltk_sent_tokenize is not None:
        try:
            sents = nltk_sent_tokenize(text)
            return [reflow(x) for x in sents[:5]]
        except Exception:
            pass
    return [reflow(x) for x in text.split('.') if reflow(x)][:5]


def aggregate_results(results: List[SimulationResult]) -> Dict[str, Any]:
    ranked = sorted(results, key=lambda item: item.score, reverse=True)
    top = ranked[: int(globals().get("TOP_K_PATHS", 8))]
    all_text = "\n".join(item.summary for item in top)
    interventions = {}
    for item in top:
        for action in item.applied_path:
            interventions[action] = interventions.get(action, 0) + 1
    memory = retrieve_memory_fragments(top[0].scenario if top else "", limit=5)
    return {
        "top_results": [asdict(item) for item in top],
        "avg_score": statistics.fmean([item.score for item in results]) if results else 0.0,
        "keyword_surface": keyword_surface(all_text),
        "sentence_surface": sentence_surface(all_text),
        "intervention_frequency": interventions,
        "memory_hits": memory,
    }

def llm_peace_brief(scenario: str, aggregate: Dict[str, Any]) -> str:
    prompt = f"""
You are producing a peaceful-resolution simulation brief.

Policy constraints:
- Only discuss de-escalation, negotiation, inspections, humanitarian access, civilian protection, and confidence-building.
- Do not provide military advice, targeting, or coercive operational planning.

Scenario:
{scenario}

Top intervention frequencies:
{json.dumps(aggregate['intervention_frequency'], indent=2)}

Keyword surface:
{aggregate['keyword_surface']}

Sentence surface:
{aggregate['sentence_surface']}

Memory hits:
{json.dumps(aggregate['memory_hits'], indent=2)}

Write a concise brief with these headings:
1. Most promising peaceful pathways
2. Why they appear resilient
3. Main fragilities
4. Low-risk next diplomatic steps
""".strip()
    brief = llm_summary(prompt)
    return brief.strip()


def deterministic_peace_brief(scenario: str, aggregate: Dict[str, Any]) -> str:
    top_freq = sorted(aggregate["intervention_frequency"].items(), key=lambda item: item[1], reverse=True)
    best_actions = ", ".join(name for name, _ in top_freq[:4]) or "No actions identified"
    risks = ", ".join(aggregate["keyword_surface"][:5]) or "limited signal"
    sents = aggregate["sentence_surface"][:3]
    return textwrap.dedent(f"""
    1. Most promising peaceful pathways
    The most recurrent stabilizing measures were: {best_actions}.

    2. Why they appear resilient
    The strongest paths consistently improved negotiation readiness, verification strength, and humanitarian access at the same time instead of optimizing only one metric.

    3. Main fragilities
    Recurring fragilities in the scan included trust deficits, verification bottlenecks, and spillover risk surfaces signaled by: {risks}.

    4. Low-risk next diplomatic steps
    Start with reciprocal, monitorable moves. Favor humanitarian corridors, backchannel talks, prisoner exchange sequencing, and third-party verification before any broader settlement claims.

    Notes
    {' '.join(sents)}
    """).strip()


# =========================
# Main execution
# =========================

def run_peace_resolution_scan(scenario: str, runs: int, months: int) -> Dict[str, Any]:
    debug_print("scan_start", f"scenario={short_text(scenario, 80)} runs={runs} months={months}")
    ensure_memory_db()
    results = []
    for run_index in range(runs):
        result = simulate_path(scenario, run_index, months)
        results.append(result)
        store_run_result({
            "scenario": result.scenario,
            "summary": result.summary,
            "score": result.score,
            "payload": asdict(result),
        })
        store_memory_fragment(scenario, "summary", result.summary, result.score)
        if run_index < 3:
            debug_print("run_result", f"run={run_index} score={result.score:.3f} summary={short_text(result.summary)}")
    aggregate = aggregate_results(results)
    llm_brief = llm_peace_brief(scenario, aggregate) if globals().get("ENABLE_LLM_SUMMARY", True) else ""
    brief = llm_brief or deterministic_peace_brief(scenario, aggregate)
    payload = {
        "scenario": scenario,
        "objective": SIMULATION_OBJECTIVE,
        "region": SIMULATION_REGION,
        "horizon_months": months,
        "runs": runs,
        "aggregate": aggregate,
        "brief": brief,
        "generated_at": time.time(),
    }
    Path = __import__('pathlib').Path
    Path(OUTFILE_JSON).write_text(json.dumps(payload, indent=2))
    Path(OUTFILE_MD).write_text(render_markdown_report(payload))
    debug_print("scan_end", f"avg_score={aggregate['avg_score']:.3f} outfile={OUTFILE_MD}")
    return payload


def render_markdown_report(payload: Dict[str, Any]) -> str:
    aggregate = payload["aggregate"]
    lines = [
        f"# Peaceful Resolution Simulation Scan",
        "",
        f"Scenario: {payload['scenario']}",
        f"Region: {payload['region']}",
        f"Runs: {payload['runs']}",
        f"Horizon (months): {payload['horizon_months']}",
        f"Average peace score: {aggregate['avg_score']:.3f}",
        "",
        "## Brief",
        payload["brief"],
        "",
        "## Keyword Surface",
    ]
    lines.extend([f"- {item}" for item in aggregate["keyword_surface"]])
    lines.append("")
    lines.append("## Top Paths")
    for idx, item in enumerate(aggregate["top_results"], 1):
        lines.append(f"### Path {idx}")
        lines.append(f"- Score: {item['score']:.3f}")
        lines.append(f"- Applied interventions: {', '.join(item['applied_path'][:10])}")
        lines.append(f"- Summary: {item['summary']}")
    return "\n".join(lines)


def main_colab() -> None:
    scenarios = globals().get("SELECTED_SCENARIOS", []) or list(DEFAULT_SCENARIO_BANK["peace_sim"])[:1]
    outputs = []
    return "\n".join(lines)
    print(json.dumps({"completed": len(outputs), "outfile": OUTFILE_MD}, indent=2))


def main_cli() -> None:
    main_colab()


if __name__ == "__main__":
    main_colab()


GGUF_PATH: /content/models/starling-lm-7b-alpha.Q4_K_M.gguf
THREADS: 2
N_GPU_LAYERS: 50


llama_context: n_ctx_per_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


[DEBUG] memory_retrieve | query=blog topics meta loop 777 mode=meta topic=blog topics hits=0
[DEBUG] memory_retrieve | query=blog topics mode=meta topic= hits=0
[DEBUG] memory_retrieve | query=Create 6 distinct prompt micro-variation items for a long-form generation system.  Topic: ... mode= topic= hits=0
[DEBUG] memory_retrieve | query=Create 6 distinct cadence constraint items for a long-form generation system.  Topic: blog... mode= topic= hits=0
[DEBUG] db_save | stage=dynamic_scaffold mode=meta topic=blog topics salience=0.92 summary=Write for Tech-curious builders and readers on blog topics with a clear, human, confident, and detailed tone from direct expert guidance with second-person phrasing when useful. Make every section earn its place, support...
[DEBUG] db_save | stage=dynamic_micro mode=meta topic=blog topics salience=0.72 summary=As you may know, generating long-form content can be quite challenging. However, there are certain key elements that you should keep in mind to 